# Machine Learning Mini-Project 4
## Regularization, Optimization, CNNs, LSTMs, and Word Embeddings

**Author:** Hossein Mirsaeidi

This notebook is organised into five parts that mirror the project guideline:

1. **Deep Feedforward Networks & Backpropagation** — forward/backward equations for a
   $2\to3\to2\to1$ network and a hand-checked single update step.
2. **Regularization** — an MLP regressor on a synthetic cubic function comparing
   *early stopping* against *dropout*.
3. **Optimization** — a 3-layer MLP on MNIST comparing *SGD (with momentum)* against *Adam*.
4. **CNN vs. LSTM** — a CNN on CIFAR-10 and an LSTM on IMDB sentiment.
5. **CBOW word embeddings** — a Continuous Bag-of-Words model, in 2-D directly and in
   10-D projected to 2-D with PCA.

All figures are exported as vector PDFs into `figures/`.

### Environment and reproducibility

We import the scientific stack and TensorFlow/Keras, fix all random seeds, enable GPU
memory growth (the development GPU is a 4 GB GTX 1650) and define a small `savefig`
helper so every plot is stored as a vector PDF.

In [ ]:
import os, sys, time, random, warnings, re
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- GPU bootstrap: make the notebook find CUDA no matter how the kernel was
#      launched or which interpreter runs it. We collect every place the
#      nvidia-*-cu12 pip wheels might live (this interpreter's site-packages AND
#      the project venv), put them on LD_LIBRARY_PATH, and ctypes-preload each
#      .so so TF's dlopen resolves them even when LD_LIBRARY_PATH was unset at
#      process start (the dynamic loader ignores later changes to it).
def _enable_cuda_libs(verbose=True):
    import glob, ctypes, site, sysconfig, importlib.util
    os.environ.pop('PYTHONPATH', None)          # drop the ROS PYTHONPATH leak

    roots = []
    # a) wherever THIS interpreter would import the `nvidia` namespace package
    try:
        spec = importlib.util.find_spec('nvidia')
        if spec and spec.submodule_search_locations:
            roots += list(spec.submodule_search_locations)
    except Exception:
        pass
    # b) this interpreter's site-packages
    cands = list(site.getsitepackages()) if hasattr(site, 'getsitepackages') else []
    cands += [site.getusersitepackages(), sysconfig.get_paths().get('purelib', '')]
    # c) the project venv (in case the kernel is some OTHER interpreter)
    here = os.getcwd()
    cands += glob.glob(os.path.join(here, 'venv', 'lib', 'python*', 'site-packages'))
    cands += glob.glob(os.path.join(here, '.venv', 'lib', 'python*', 'site-packages'))
    roots += [os.path.join(p, 'nvidia') for p in cands if p]

    lib_dirs = sorted({os.path.dirname(f)
                       for r in roots if os.path.isdir(r)
                       for f in glob.glob(os.path.join(r, '*', 'lib', '*.so*'))})
    if not lib_dirs:
        if verbose:
            print('[gpu-bootstrap] no nvidia-*-cu12 libs found for', sys.executable)
        return False
    os.environ['LD_LIBRARY_PATH'] = ':'.join(
        lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')]).strip(':')
    sos = sorted({f for d in lib_dirs for f in glob.glob(os.path.join(d, '*.so*'))})
    for _ in range(3):
        for so in sos:
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
    if verbose:
        print('[gpu-bootstrap] preloaded', len(sos), 'CUDA libs from', len(lib_dirs), 'dirs')
    return True

print('[gpu-bootstrap] interpreter:', sys.executable)
_enable_cuda_libs()
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '1')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---- reproducibility -------------------------------------------------------
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- GPU memory growth (4 GB card) -----------------------------------------
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print('memory growth:', e)

# ---- figure export ---------------------------------------------------------
FIGDIR = 'figures'
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox': 'tight', 'font.size': 11})
sns.set_theme(style='whitegrid', context='notebook')

def savefig(name):
    path = os.path.join(FIGDIR, name)
    plt.savefig(path)
    print('saved', path)

# ---- global training configuration (epochs follow the specification) -------
REG_EPOCHS   = int(os.environ.get('REG_EPOCHS',   200))
MNIST_EPOCHS = int(os.environ.get('MNIST_EPOCHS', 20))
CNN_EPOCHS   = int(os.environ.get('CNN_EPOCHS',   50))
LSTM_EPOCHS  = int(os.environ.get('LSTM_EPOCHS',  20))
CBOW_EPOCHS  = int(os.environ.get('CBOW_EPOCHS',  500))

print('TensorFlow', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

---
# Part 1 — Deep Feedforward Networks and Backpropagation

We study a fully connected network
$$\text{Input }(2)\;\to\;\text{Hidden }(3,\ \mathrm{ReLU})\;\to\;\text{Hidden }(2,\ \mathrm{ReLU})\;\to\;\text{Output }(1,\ \text{linear}).$$

Complete solution is provided in the handwritten report.

---
# Part 2 — Regularization for Deep Learning

### 2.1 Data generation

We sample $N=1000$ points uniformly from $[-10,10]^2$, evaluate
$f(x_1,x_2)=x_1^3-x_1^2+2x_2^2+3x_1x_2+5$, add uniform noise $\varepsilon\sim\mathcal{U}(-1,1)$,
and split 80/20. Because the target spans a very wide range (roughly $\pm10^3$), we
**standardise** the inputs and the target using statistics from the training set only;
predictions are mapped back to the original units before we report the MSE.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def f_true(x1, x2):
    return x1**3 - x1**2 + 2*x2**2 + 3*x1*x2 + 5

rng = np.random.default_rng(SEED)
N = 1000
X = rng.uniform(-10, 10, size=(N, 2))
y_clean = f_true(X[:, 0], X[:, 1])
eps = rng.uniform(-1, 1, size=N)
y = y_clean + eps

X_tr_raw, X_te_raw, y_tr_raw, y_te_raw = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

x_scaler = StandardScaler().fit(X_tr_raw)
y_scaler = StandardScaler().fit(y_tr_raw.reshape(-1, 1))
Xtr = x_scaler.transform(X_tr_raw); Xte = x_scaler.transform(X_te_raw)
ytr = y_scaler.transform(y_tr_raw.reshape(-1, 1)).ravel()
yte = y_scaler.transform(y_te_raw.reshape(-1, 1)).ravel()
print('train', Xtr.shape, ' test', Xte.shape)

def test_mse_original_units(model):
    # predict in standardized space, invert the target scaling, compare to raw targets
    pred = y_scaler.inverse_transform(model.predict(Xte, verbose=0)).ravel()
    return float(np.mean((pred - y_te_raw) ** 2)), pred

### 2.2 / 2.3 Model, baseline training and evaluation

The MLP has two hidden layers of 10 ReLU units and a single linear output, trained with
Adam and MSE for up to 200 epochs with a 10% validation split.

In [ ]:
def build_reg_mlp(dropout=0.0):
    m = keras.Sequential(name='reg_mlp')
    m.add(keras.Input(shape=(2,)))
    m.add(layers.Dense(10, activation='relu'))
    if dropout > 0: m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation='relu'))
    if dropout > 0: m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation='linear'))
    m.compile(optimizer='adam', loss='mse')
    return m

keras.utils.set_random_seed(SEED)
model_base = build_reg_mlp()
t0 = time.time()
h_base = model_base.fit(Xtr, ytr, validation_split=0.1,
                        epochs=REG_EPOCHS, batch_size=32, verbose=0)
t_base = time.time() - t0
mse_base, pred_base = test_mse_original_units(model_base)
print(f'baseline: {t_base:.1f}s, epochs={len(h_base.history["loss"])}, '
      f'test MSE (original units) = {mse_base:.3f}')

In [ ]:
# Loss curves + true-vs-predicted scatter for the baseline
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(h_base.history['loss'], label='training loss')
ax[0].plot(h_base.history['val_loss'], label='validation loss')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('MSE (standardized)')
ax[0].set_title('Baseline MLP — learning curves'); ax[0].legend()

ax[1].scatter(y_te_raw, pred_base, s=12, alpha=0.6)
lims = [y_te_raw.min(), y_te_raw.max()]
ax[1].plot(lims, lims, 'r--', lw=1)
ax[1].set_xlabel('true y'); ax[1].set_ylabel('predicted y')
ax[1].set_title('Baseline MLP — test predictions')
savefig('p2_baseline.pdf'); plt.show()

### 2.4 Regularization strategies

We retrain the same architecture twice: (a) with **early stopping** (patience 10,
restoring the best weights) and (b) with **dropout** ($p=0.5$ after each hidden layer,
full 200 epochs), then compare training time and test MSE.

In [ ]:
# (a) Early stopping
keras.utils.set_random_seed(SEED)
model_es = build_reg_mlp()
es_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                      restore_best_weights=True)
t0 = time.time()
h_es = model_es.fit(Xtr, ytr, validation_split=0.1, epochs=REG_EPOCHS,
                    batch_size=32, verbose=0, callbacks=[es_cb])
t_es = time.time() - t0
stop_epoch = len(h_es.history['loss'])
mse_es, _ = test_mse_original_units(model_es)
print(f'early stopping: stopped at epoch {stop_epoch}, {t_es:.1f}s, test MSE = {mse_es:.3f}')

In [ ]:
# (b) Dropout, full 200 epochs
keras.utils.set_random_seed(SEED)
model_do = build_reg_mlp(dropout=0.5)
t0 = time.time()
h_do = model_do.fit(Xtr, ytr, validation_split=0.1, epochs=REG_EPOCHS,
                    batch_size=32, verbose=0)
t_do = time.time() - t0
mse_do, _ = test_mse_original_units(model_do)
print(f'dropout: {t_do:.1f}s, test MSE = {mse_do:.3f}')

In [ ]:
# (c) Comparison table + bar charts
comp = pd.DataFrame({
    'strategy':  ['baseline (200)', 'early stopping', 'dropout (200)'],
    'epochs':    [len(h_base.history['loss']), stop_epoch, len(h_do.history['loss'])],
    'train_time_s': [t_base, t_es, t_do],
    'test_MSE':  [mse_base, mse_es, mse_do],
})
display(comp.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(comp['strategy'], comp['train_time_s'], color='steelblue')
ax[0].set_ylabel('training time (s)'); ax[0].set_title('Training time')
ax[0].tick_params(axis='x', rotation=15)
ax[1].bar(comp['strategy'], comp['test_MSE'], color='indianred')
ax[1].set_ylabel('test MSE (original units)'); ax[1].set_title('Test MSE')
ax[1].tick_params(axis='x', rotation=15)
savefig('p2_comparison.pdf'); plt.show()

**Discussion.** Early stopping halts well before 200 epochs, so it is the cheapest option,
and on this small task it usually reaches a test MSE close to (or better than) the fully
trained baseline. Dropout at $p=0.5$ is aggressive for a network with only ten units per
layer: it adds a lot of noise, needs the whole training budget, and typically ends with a
higher test MSE. For this regression problem we therefore **recommend early stopping**.

---
# Part 3 — Optimization for Training Deep Networks

### 3a) Update rules: SGD, Momentum, Adam

Let $g_t=\nabla_\theta\mathcal{L}(\theta_{t})$ be the gradient at step $t$.

**SGD:** $\quad \theta_{t+1}=\theta_t-\eta\,g_t.$

**Momentum:** $\quad v_t=\mu\,v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta\,v_t,$
where $\mu$ (e.g. 0.9) accumulates a velocity that damps oscillations and speeds up
progress along consistent directions.

**Adam:** first and second moment estimates with bias correction,
$$
m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\qquad v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2,
$$
$$
\hat m_t=\frac{m_t}{1-\beta_1^{t}},\quad \hat v_t=\frac{v_t}{1-\beta_2^{t}},\qquad
\theta_{t+1}=\theta_t-\eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$
Adam adapts a per-parameter step size, combining momentum with RMS normalisation.

### 3.1–3.3 MNIST: SGD vs. Adam

We load MNIST, scale pixels to $[0,1]$, one-hot encode the labels, and train two identical
$784\to128\to64\to10$ MLPs for 20 epochs — one with SGD (lr 0.01, momentum 0.9) and one
with Adam (defaults) — then compare their learning curves and test accuracy.

In [ ]:
(xm_tr, ym_tr), (xm_te, ym_te) = keras.datasets.mnist.load_data()
xm_tr = xm_tr.reshape(-1, 784).astype('float32') / 255.0
xm_te = xm_te.reshape(-1, 784).astype('float32') / 255.0
Ym_tr = keras.utils.to_categorical(ym_tr, 10)
Ym_te = keras.utils.to_categorical(ym_te, 10)
print('MNIST:', xm_tr.shape, xm_te.shape)

def build_mnist_mlp():
    return keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')])

def train_mnist(optimizer):
    keras.utils.set_random_seed(SEED)
    m = build_mnist_mlp()
    m.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    h = m.fit(xm_tr, Ym_tr, validation_split=0.1, epochs=MNIST_EPOCHS,
              batch_size=128, verbose=0)
    dt = time.time() - t0
    test_acc = m.evaluate(xm_te, Ym_te, verbose=0)[1]
    return h, test_acc, dt

h_sgd,  acc_sgd,  t_sgd  = train_mnist(keras.optimizers.SGD(learning_rate=0.01, momentum=0.9))
h_adam, acc_adam, t_adam = train_mnist(keras.optimizers.Adam())
print(f'SGD : test acc = {acc_sgd:.4f}  ({t_sgd:.1f}s)')
print(f'Adam: test acc = {acc_adam:.4f}  ({t_adam:.1f}s)')

In [ ]:
# Overlaid learning curves (accuracy and loss)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(h_sgd.history['val_accuracy'],  label='SGD val acc')
ax[0].plot(h_adam.history['val_accuracy'], label='Adam val acc')
ax[0].plot(h_sgd.history['accuracy'],  '--', alpha=0.6, label='SGD train acc')
ax[0].plot(h_adam.history['accuracy'], '--', alpha=0.6, label='Adam train acc')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy')
ax[0].set_title('MNIST accuracy'); ax[0].legend(fontsize=8)

ax[1].plot(h_sgd.history['val_loss'],  label='SGD val loss')
ax[1].plot(h_adam.history['val_loss'], label='Adam val loss')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('loss')
ax[1].set_title('MNIST loss'); ax[1].legend(fontsize=8)
savefig('p3_sgd_vs_adam.pdf'); plt.show()

print(f'Final test accuracy  ->  SGD: {acc_sgd:.4f}   Adam: {acc_adam:.4f}')

**Comment.** Adam usually reaches a low loss in the first few epochs, whereas SGD climbs
more gradually. Both converge to a similar final test accuracy on this easy task, but Adam
is faster and smoother early on; SGD with momentum can be slightly more stable near the end
and is less prone to the small late-epoch fluctuations sometimes seen with Adam.

---
# Part 4 — CNN vs. LSTM on Benchmark Datasets

## 4.1 CNN on CIFAR-10

We load CIFAR-10, scale pixels to $[0,1]$ and one-hot encode the ten classes. The network
follows the specified architecture and is trained with Adam and categorical cross-entropy
for 50 epochs (batch 64, 10% validation).

In [ ]:
(xc_tr, yc_tr), (xc_te, yc_te) = keras.datasets.cifar10.load_data()
xc_tr = xc_tr.astype('float32') / 255.0
xc_te = xc_te.astype('float32') / 255.0
Yc_tr = keras.utils.to_categorical(yc_tr, 10)
Yc_te = keras.utils.to_categorical(yc_te, 10)
cifar_classes = ['airplane','automobile','bird','cat','deer',
                 'dog','frog','horse','ship','truck']
print('CIFAR-10:', xc_tr.shape, xc_te.shape)

def build_cnn():
    m = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

keras.utils.set_random_seed(SEED)
cnn = build_cnn()
cnn.summary()

In [ ]:
t0 = time.time()
h_cnn = cnn.fit(xc_tr, Yc_tr, validation_split=0.1,
                epochs=CNN_EPOCHS, batch_size=64, verbose=2)
t_cnn = time.time() - t0
cnn_test_acc = cnn.evaluate(xc_te, Yc_te, verbose=0)[1]
print(f'CNN test accuracy = {cnn_test_acc:.4f}   training time = {t_cnn/60:.1f} min')
try:
    print(f"peak GPU memory: {tf.config.experimental.get_memory_info('GPU:0')['peak']/1e9:.2f} GB")
except Exception:
    pass

In [ ]:
# Learning curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(h_cnn.history['accuracy'], label='train')
ax[0].plot(h_cnn.history['val_accuracy'], label='val')
ax[0].set_title('CNN accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h_cnn.history['loss'], label='train')
ax[1].plot(h_cnn.history['val_loss'], label='val')
ax[1].set_title('CNN loss'); ax[1].set_xlabel('epoch'); ax[1].legend()
savefig('p4_cnn_curves.pdf'); plt.show()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cnn_probs = cnn.predict(xc_te, verbose=0)
y_pred = cnn_probs.argmax(1); y_true = Yc_te.argmax(1)
cm = confusion_matrix(y_true, y_pred)
_off = cm.copy(); np.fill_diagonal(_off, 0)
_top = sorted(((int(_off[a, b]), cifar_classes[a], cifar_classes[b])
               for a in range(10) for b in range(10)), reverse=True)[:3]
print('top confusions (true -> pred, count):', [(t, p, n) for n, t, p in _top])
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=cifar_classes).plot(
    ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
ax.set_title('CIFAR-10 confusion matrix')
savefig('p4_cnn_confusion.pdf'); plt.show()

In [ ]:
# ROC curves, one-vs-rest
from sklearn.metrics import roc_curve, auc
fpr, tpr, roc_auc = {}, {}, {}
for i in range(10):
    fpr[i], tpr[i], _ = roc_curve(Yc_te[:, i], cnn_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
fpr_micro, tpr_micro, _ = roc_curve(Yc_te.ravel(), cnn_probs.ravel())
auc_micro = auc(fpr_micro, tpr_micro)
print('per-class AUC:', {cifar_classes[i]: round(roc_auc[i], 3) for i in range(10)})
print(f'micro-average AUC = {auc_micro:.3f}')

fig, ax = plt.subplots(figsize=(7.5, 6.5))
for i in range(10):
    ax.plot(fpr[i], tpr[i], lw=1, label=f'{cifar_classes[i]} ({roc_auc[i]:.2f})')
ax.plot(fpr_micro, tpr_micro, 'k--', lw=2, label=f'micro-average ({auc_micro:.2f})')
ax.plot([0, 1], [0, 1], color='grey', lw=0.8, ls=':')
ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate')
ax.set_title('CIFAR-10 ROC (one-vs-rest)'); ax.legend(fontsize=8, loc='lower right')
savefig('p4_cnn_roc.pdf'); plt.show()

In [ ]:
# A few misclassified images
mis = np.where(y_pred != y_true)[0][:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), mis):
    ax.imshow(xc_te[idx]); ax.axis('off')
    ax.set_title(f'T:{cifar_classes[y_true[idx]]}\nP:{cifar_classes[y_pred[idx]]}', fontsize=9)
fig.suptitle('CIFAR-10 misclassified examples')
savefig('p4_cnn_misclassified.pdf'); plt.show()

## 4.2 LSTM on IMDB sentiment

Each review is an integer sequence over the 20,000 most frequent words; rarer words map to
`<UNK>`. We pad/truncate to 200 tokens (zeros prepended, extra words at the end dropped).
The model is Embedding(20000, 128) → LSTM(128) → Dense(64, ReLU) → Dropout(0.5) →
Dense(1, sigmoid), trained with Adam and binary cross-entropy for 20 epochs.

In [ ]:
VOCAB, MAXLEN = 20000, 200
(xi_tr, yi_tr), (xi_te, yi_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
xi_tr = keras.utils.pad_sequences(xi_tr, maxlen=MAXLEN, padding='pre', truncating='post')
xi_te = keras.utils.pad_sequences(xi_te, maxlen=MAXLEN, padding='pre', truncating='post')
print('IMDB:', xi_tr.shape, xi_te.shape)

def build_lstm():
    m = keras.Sequential([
        keras.Input(shape=(MAXLEN,)),
        layers.Embedding(VOCAB, 128),
        layers.LSTM(128),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

keras.utils.set_random_seed(SEED)
lstm = build_lstm()
lstm.summary()

In [ ]:
t0 = time.time()
h_lstm = lstm.fit(xi_tr, yi_tr, validation_split=0.1,
                  epochs=LSTM_EPOCHS, batch_size=64, verbose=2)
t_lstm = time.time() - t0
print(f'LSTM training time = {t_lstm/60:.1f} min')
try:
    print(f"peak GPU memory: {tf.config.experimental.get_memory_info('GPU:0')['peak']/1e9:.2f} GB")
except Exception:
    pass

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
lstm_probs = lstm.predict(xi_te, verbose=0).ravel()
lstm_pred = (lstm_probs >= 0.5).astype(int)
print(f'accuracy  = {accuracy_score(yi_te, lstm_pred):.4f}')
print(f'precision = {precision_score(yi_te, lstm_pred):.4f}')
print(f'recall    = {recall_score(yi_te, lstm_pred):.4f}')
print(f'F1-score  = {f1_score(yi_te, lstm_pred):.4f}')

# learning curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(h_lstm.history['accuracy'], label='train')
ax[0].plot(h_lstm.history['val_accuracy'], label='val')
ax[0].set_title('IMDB LSTM accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h_lstm.history['loss'], label='train')
ax[1].plot(h_lstm.history['val_loss'], label='val')
ax[1].set_title('IMDB LSTM loss'); ax[1].set_xlabel('epoch'); ax[1].legend()
savefig('p4_lstm_curves.pdf'); plt.show()

In [ ]:
# Decode and show a few misclassified reviews
word_index = keras.datasets.imdb.get_word_index()
index_word = {v + 3: k for k, v in word_index.items()}
index_word[0] = '<PAD>'; index_word[1] = '<START>'; index_word[2] = '<UNK>'; index_word[3] = '<UNUSED>'
def decode(seq):
    return ' '.join(index_word.get(i, '?') for i in seq if i != 0)

mis = np.where(lstm_pred != yi_te)[0][:4]
for idx in mis:
    sent = 'positive' if yi_te[idx] == 1 else 'negative'
    print(f'TRUE={sent}  PRED_prob={lstm_probs[idx]:.2f}')
    print(decode(xi_te[idx])[:400], '...\n')

---
# Part 5 — Continuous Bag-of-Words (CBOW) Word Embeddings

We train a CBOW model on a ~1000-word text about machine learning / neural networks
(`data/cbow_corpus.txt`). Given a context window of two words on each side, the model
predicts the centre word through a softmax over the vocabulary. We first learn 2-D
embeddings directly, then learn 10-D embeddings and project them to 2-D with PCA.

In [ ]:
# Preprocess corpus and build CBOW (context -> center) training pairs
text = open('data/cbow_corpus.txt', encoding='utf-8').read().lower()
tokens = re.findall(r'[a-z]+', text)
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)
seq = [word2idx[w] for w in tokens]

WIN = 2
contexts, targets = [], []
for i in range(WIN, len(seq) - WIN):
    contexts.append([seq[i-2], seq[i-1], seq[i+1], seq[i+2]])
    targets.append(seq[i])
contexts = np.array(contexts); targets = np.array(targets)

from collections import Counter
freq = Counter(tokens)
print(f'tokens={len(tokens)}, vocabulary V={V}, training pairs={len(targets)}')

def build_cbow(dim):
    inp = keras.Input(shape=(4,))
    emb = layers.Embedding(V, dim, name='cbow_embedding')(inp)
    avg = layers.GlobalAveragePooling1D()(emb)
    out = layers.Dense(V, activation='softmax')(avg)
    m = keras.Model(inp, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def scatter_embeddings(E2, title, fname, n_labels=45):
    top = [w for w, _ in freq.most_common(n_labels)]
    fig, ax = plt.subplots(figsize=(11, 9))
    ax.scatter(E2[:, 0], E2[:, 1], s=14, alpha=0.4, color='steelblue')
    for w in top:
        i = word2idx[w]
        ax.annotate(w, (E2[i, 0], E2[i, 1]), fontsize=8, alpha=0.9)
    ax.set_title(title); ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
    savefig(fname); plt.show()

### 5.1–5.3 Direct 2-D embedding

In [ ]:
keras.utils.set_random_seed(SEED)
cbow2 = build_cbow(2)
hist2 = cbow2.fit(contexts, targets, epochs=CBOW_EPOCHS, batch_size=64, verbose=0)
print(f'2-D CBOW final loss = {hist2.history["loss"][-1]:.3f}')
E2 = cbow2.get_layer('cbow_embedding').get_weights()[0]   # (V, 2)
def nearest(E, w, k=5):
    v = E[word2idx[w]]
    sims = (E @ v) / (np.linalg.norm(E, axis=1) * np.linalg.norm(v) + 1e-9)
    return [idx2word[i] for i in np.argsort(-sims)[1:k + 1]]
print('2-D nearest neighbours:')
for w in ['network', 'learning', 'data', 'neural', 'model']:
    if w in word2idx: print(f'  {w:10s} -> {nearest(E2, w)}')
scatter_embeddings(E2, 'CBOW 2-D embeddings (direct)', 'p5_cbow_2d.pdf')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(hist2.history['loss']) + 1), hist2.history['loss'], label='training loss')
ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.set_title('CBOW 2-D training loss')
ax.legend()
savefig('p5_cbow_2d_loss.pdf')
plt.show()

**Discussion.** Because the corpus is small, only frequent domain words obtain
well-trained vectors. Words that occur in similar contexts — e.g. *network*, *neural*,
*layer*, *neuron*, or *learning*, *training*, *data* — tend to sit closer together, while
function words spread out. The 2-D space is tight, so clusters are only roughly separated.

For readability the scatter plots show all $V$ points but label only the most frequent words; the full vocabulary is small enough that the labelled subset is representative of the clusters.

### 5.4 10-D embedding with PCA reduction

In [ ]:
from sklearn.decomposition import PCA
keras.utils.set_random_seed(SEED)
cbow10 = build_cbow(10)
hist10 = cbow10.fit(contexts, targets, epochs=CBOW_EPOCHS, batch_size=64, verbose=0)
print(f'10-D CBOW final loss = {hist10.history["loss"][-1]:.3f}')
E10 = cbow10.get_layer('cbow_embedding').get_weights()[0]  # (V, 10)
E10_2d = PCA(n_components=2, random_state=SEED).fit_transform(E10)
print('10-D nearest neighbours (full space):')
for w in ['network', 'learning', 'data', 'neural', 'model']:
    if w in word2idx: print(f'  {w:10s} -> {nearest(E10, w)}')
scatter_embeddings(E10_2d, 'CBOW 10-D embeddings projected to 2-D (PCA)', 'p5_cbow_10d_pca.pdf')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(hist10.history['loss']) + 1), hist10.history['loss'], label='training loss')
ax.set_xlabel('epoch')
ax.set_ylabel('loss')
ax.set_title('CBOW 10-D training loss')
ax.legend()
savefig('p5_cbow_10d_loss.pdf')
plt.show()

**Comparison.** The 10-D model has more room to separate word senses, so after PCA the
related-word groups are usually a little cleaner and better spread than in the direct 2-D
model, even though the overall layout is similar. The direct 2-D model is forced to pack
everything into two axes, which can merge unrelated words; PCA of the richer 10-D space
keeps the two most informative directions and often gives tighter clusters. Both are
limited by the tiny corpus.